# Preprocess

In [ ]:
# RIGHT AFTER (RE)STARTING THE KERNEL, RUN THIS CELL BEFORE LOADING A CHECKPOINT

import numpy as np
import pandas as pd
from helpers.pathlib import PATH_DATA, get_path_site_checkpoint

from ata_pipeline1.helpers.enums import FieldNew, FieldSnowplow
from ata_pipeline1.preprocessors import (
    AddFieldDeviceIsMobile,
    AddFieldEventParentId,
    AddFieldFormSubmitIsNewsletter,
    AddFieldLeadsToNewsletterConversion,
    AddFieldMaxScrollDepth,
    AddFieldSessionEventIndex,
    AddFieldsPageType,
    AggregatePageActivities,
    ConvertFieldTypes,
    DeleteRowsOutlier,
    ParseInternalReferrerUrls,
    ReclassifyInternalReferrals,
    ReclassifyNullReferrals,
    ReplaceValues,
    SetRowIndex,
    SortFieldTimestamp,
)
from ata_pipeline1.site import AFRO_LA, DALLAS_FREE_PRESS, OPEN_VALLEJO, THE_19TH

CSV_PREFIX = "221103-230125"

In [ ]:
# Read data
df = pd.read_csv(PATH_DATA / f"{CSV_PREFIX}.csv")

In [ ]:
df.shape

## Convert types

In [ ]:
df = ConvertFieldTypes()(df)

## Replace null values in `pp_yoffset_max` 

In [ ]:
df = ReplaceValues(replace_what=np.nan, replace_with=0, fields={FieldSnowplow.PP_YOFFSET_MAX})(df)

## Remove outliers

In [ ]:
# This can just be done as part of the SQL fetch
df = DeleteRowsOutlier(
    bounds={
        # Next 4 pairs of bounds reflect practical minimum resolution & resolution of an 8K screen
        FieldSnowplow.BR_VIEWHEIGHT: (96, 4320),
        FieldSnowplow.BR_VIEWWIDTH: (128, 7680),
        FieldSnowplow.DVCE_SCREENHEIGHT: (96, 4320),
        FieldSnowplow.DVCE_SCREENWIDTH: (128, 7680),
        # doc_height lower bound follows lowest viewport height, upper bound follows what the height
        # of an OpenVallejo longform investigation could look like on a small-screen-width device
        FieldSnowplow.DOC_HEIGHT: (96, 30000),
        # No pp_yoffset_max lower bound, upper bound follows that of doc_height
        FieldSnowplow.PP_YOFFSET_MAX: (None, 30000),
    }
)(df)

## Replace negative `pp_yoffset_max` values

## Sort by ascending timestamp

In [ ]:
df = SortFieldTimestamp()(df)

## Check if device is mobile

In [ ]:
df = AddFieldDeviceIsMobile()(df)

## Get individual site DataFrames

In [ ]:
df_afla = df.query(f"site_name == '{AFRO_LA.name}'")
df_dfp = df.query(f"site_name == '{DALLAS_FREE_PRESS.name}'")
df_ov = df.query(f"site_name == '{OPEN_VALLEJO.name}'")
df_19 = df.query(f"site_name == '{THE_19TH.name}'")

In [ ]:
assert df_afla.derived_tstamp.is_monotonic_increasing
assert df_dfp.derived_tstamp.is_monotonic_increasing
assert df_ov.derived_tstamp.is_monotonic_increasing
assert df_19.derived_tstamp.is_monotonic_increasing

## Add max scroll depths

In [ ]:
df_afla = AddFieldMaxScrollDepth()(df_afla)
df_dfp = AddFieldMaxScrollDepth()(df_dfp)
df_ov = AddFieldMaxScrollDepth()(df_ov)
df_19 = AddFieldMaxScrollDepth()(df_19)

In [ ]:
# CHECKPOINT 1

df_afla.to_pickle(get_path_site_checkpoint(CSV_PREFIX, 1, AFRO_LA.name))
df_dfp.to_pickle(get_path_site_checkpoint(CSV_PREFIX, 1, DALLAS_FREE_PRESS.name))
df_ov.to_pickle(get_path_site_checkpoint(CSV_PREFIX, 1, OPEN_VALLEJO.name))
df_19.to_pickle(get_path_site_checkpoint(CSV_PREFIX, 1, THE_19TH.name))

## Set event ID as row index

In [ ]:
# Read data from CHECKPOINT 1
df_afla = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 1, AFRO_LA.name))
df_dfp = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 1, DALLAS_FREE_PRESS.name))
df_ov = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 1, OPEN_VALLEJO.name))
df_19 = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 1, THE_19TH.name))

In [ ]:
df_afla = SetRowIndex([FieldSnowplow.EVENT_ID], sort_index=False)(df_afla)
df_dfp = SetRowIndex([FieldSnowplow.EVENT_ID], sort_index=False)(df_dfp)
df_ov = SetRowIndex([FieldSnowplow.EVENT_ID], sort_index=False)(df_ov)
df_19 = SetRowIndex([FieldSnowplow.EVENT_ID], sort_index=False)(df_19)

## Add newsletter-signup validation

In [ ]:
df_afla = AddFieldFormSubmitIsNewsletter(site_newsletter_signup_validator=AFRO_LA.newsletter_signup_validator)(df_afla)
df_dfp = AddFieldFormSubmitIsNewsletter(site_newsletter_signup_validator=DALLAS_FREE_PRESS.newsletter_signup_validator)(
    df_dfp
)
df_ov = AddFieldFormSubmitIsNewsletter(site_newsletter_signup_validator=OPEN_VALLEJO.newsletter_signup_validator)(df_ov)
df_19 = AddFieldFormSubmitIsNewsletter(site_newsletter_signup_validator=THE_19TH.newsletter_signup_validator)(df_19)

## Add parent-event column

In [ ]:
assert df_afla.derived_tstamp.is_monotonic_increasing
assert df_dfp.derived_tstamp.is_monotonic_increasing
assert df_ov.derived_tstamp.is_monotonic_increasing
assert df_19.derived_tstamp.is_monotonic_increasing

In [ ]:
df_afla = AddFieldEventParentId()(df_afla)
df_dfp = AddFieldEventParentId()(df_dfp)
df_ov = AddFieldEventParentId()(df_ov)
df_19 = AddFieldEventParentId()(df_19)

In [ ]:
assert df_afla.shape[0] == df.query(f"site_name == '{AFRO_LA.name}'").shape[0]
assert df_dfp.shape[0] == df.query(f"site_name == '{DALLAS_FREE_PRESS.name}'").shape[0]
assert df_ov.shape[0] == df.query(f"site_name == '{OPEN_VALLEJO.name}'").shape[0]
assert df_19.shape[0] == df.query(f"site_name == '{THE_19TH.name}'").shape[0]

In [ ]:
assert df_afla.derived_tstamp.is_monotonic_increasing
assert df_dfp.derived_tstamp.is_monotonic_increasing
assert df_ov.derived_tstamp.is_monotonic_increasing
assert df_19.derived_tstamp.is_monotonic_increasing

In [ ]:
def assert_first_ts_is_also_min_ts(df):
    assert (
        df.groupby(FieldNew.EVENT_PARENT_ID)[FieldSnowplow.DERIVED_TSTAMP].first()
        == df.groupby(FieldNew.EVENT_PARENT_ID)[FieldSnowplow.DERIVED_TSTAMP].min()
    ).all()


assert_first_ts_is_also_min_ts(df_afla)
assert_first_ts_is_also_min_ts(df_dfp)
assert_first_ts_is_also_min_ts(df_ov)
assert_first_ts_is_also_min_ts(df_19)

In [ ]:
# CHECKPOINT 2

df_afla.to_pickle(get_path_site_checkpoint(CSV_PREFIX, 2, AFRO_LA.name))
df_dfp.to_pickle(get_path_site_checkpoint(CSV_PREFIX, 2, DALLAS_FREE_PRESS.name))
df_ov.to_pickle(get_path_site_checkpoint(CSV_PREFIX, 2, OPEN_VALLEJO.name))
df_19.to_pickle(get_path_site_checkpoint(CSV_PREFIX, 2, THE_19TH.name))

## Aggregate page activities

In [ ]:
# Read data from CHECKPOINT 2
df_afla = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 2, AFRO_LA.name))
df_dfp = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 2, DALLAS_FREE_PRESS.name))
df_ov = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 2, OPEN_VALLEJO.name))
df_19 = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 2, THE_19TH.name))

In [ ]:
assert df_afla.derived_tstamp.is_monotonic_increasing
assert df_dfp.derived_tstamp.is_monotonic_increasing
assert df_ov.derived_tstamp.is_monotonic_increasing
assert df_19.derived_tstamp.is_monotonic_increasing

In [ ]:
df_afla_agg = AggregatePageActivities()(df_afla)
df_dfp_agg = AggregatePageActivities()(df_dfp)
df_ov_agg = AggregatePageActivities()(df_ov)
df_19_agg = AggregatePageActivities()(df_19)

### Dwell-time gut checks

X-axis unit is in seconds.

In [ ]:
# AfroLA
print(df_afla_agg.dwell_secs.describe())
# df_afla_agg.dwell_secs.plot.box(vert=False)

In [ ]:
# DFP
print(df_dfp_agg.dwell_secs.describe())
# df_dfp_agg.dwell_secs.plot.box(vert=False)

In [ ]:
# OpenVallejo
print(df_ov_agg.dwell_secs.describe())
# df_ov_agg.dwell_secs.plot.box(vert=False)

In [ ]:
# The 19th*
print(df_19_agg.dwell_secs.describe())
# df_19_agg.dwell_secs.plot.box(vert=False)

In [ ]:
# CHECKPOINT 3

df_afla_agg.to_pickle(get_path_site_checkpoint(CSV_PREFIX, 3, AFRO_LA.name))
df_dfp_agg.to_pickle(get_path_site_checkpoint(CSV_PREFIX, 3, DALLAS_FREE_PRESS.name))
df_ov_agg.to_pickle(get_path_site_checkpoint(CSV_PREFIX, 3, OPEN_VALLEJO.name))
df_19_agg.to_pickle(get_path_site_checkpoint(CSV_PREFIX, 3, THE_19TH.name))

## Classify page type

In [ ]:
# Read data from CHECKPOINT 3
df_afla_agg = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 3, AFRO_LA.name))
df_dfp_agg = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 3, DALLAS_FREE_PRESS.name))
df_ov_agg = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 3, OPEN_VALLEJO.name))
df_19_agg = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 3, THE_19TH.name))

In [ ]:
df_afla_agg = AddFieldsPageType(site_page_type_classifier=AFRO_LA.page_type_classifier)(df_afla_agg)

In [ ]:
df_dfp_agg = AddFieldsPageType(site_page_type_classifier=DALLAS_FREE_PRESS.page_type_classifier)(df_dfp_agg)

In [ ]:
df_ov_agg = AddFieldsPageType(site_page_type_classifier=OPEN_VALLEJO.page_type_classifier)(df_ov_agg)

In [ ]:
# This takes about 4 minutes and 2 GB of memory to run.
# If that's a problem maybe look into batching (as part of the Preprocessing base class so it's reusable)?
df_19_agg = AddFieldsPageType(site_page_type_classifier=THE_19TH.page_type_classifier)(df_19_agg)

In [ ]:
# CHECKPOINT 4

df_afla_agg.to_pickle(get_path_site_checkpoint(CSV_PREFIX, 4, AFRO_LA.name))
df_dfp_agg.to_pickle(get_path_site_checkpoint(CSV_PREFIX, 4, DALLAS_FREE_PRESS.name))
df_ov_agg.to_pickle(get_path_site_checkpoint(CSV_PREFIX, 4, OPEN_VALLEJO.name))
df_19_agg.to_pickle(get_path_site_checkpoint(CSV_PREFIX, 4, THE_19TH.name))

## Reclassify null or empty referrer media

In [ ]:
# Read data from CHECKPOINT 4
df_afla_agg = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 4, AFRO_LA.name))
df_dfp_agg = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 4, DALLAS_FREE_PRESS.name))
df_ov_agg = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 4, OPEN_VALLEJO.name))
df_19_agg = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 4, THE_19TH.name))

In [ ]:
df_afla_agg = ReplaceValues(replace_what=[np.nan], replace_with=[None], fields={FieldSnowplow.REFR_MEDIUM})(df_afla_agg)
df_dfp_agg = ReplaceValues(replace_what=[np.nan], replace_with=[None], fields={FieldSnowplow.REFR_MEDIUM})(df_dfp_agg)
df_ov_agg = ReplaceValues(replace_what=[np.nan], replace_with=[None], fields={FieldSnowplow.REFR_MEDIUM})(df_ov_agg)
df_19_agg = ReplaceValues(replace_what=[np.nan], replace_with=[None], fields={FieldSnowplow.REFR_MEDIUM})(df_19_agg)

In [ ]:
df_afla_agg = ReclassifyNullReferrals(site_domains=AFRO_LA.domains)(df_afla_agg)
df_dfp_agg = ReclassifyNullReferrals(site_domains=DALLAS_FREE_PRESS.domains)(df_dfp_agg)
df_ov_agg = ReclassifyNullReferrals(site_domains=OPEN_VALLEJO.domains)(df_ov_agg)
df_19_agg = ReclassifyNullReferrals(site_domains=THE_19TH.domains)(df_19_agg)

## Reclassify internal referrer media

In [ ]:
df_afla_agg = ReplaceValues(replace_what=[np.nan, None], replace_with="", fields={FieldSnowplow.PAGE_REFERRER})(
    df_afla_agg
)
df_dfp_agg = ReplaceValues(replace_what=[np.nan, None], replace_with="", fields={FieldSnowplow.PAGE_REFERRER})(
    df_dfp_agg
)
df_ov_agg = ReplaceValues(replace_what=[np.nan, None], replace_with="", fields={FieldSnowplow.PAGE_REFERRER})(df_ov_agg)
df_19_agg = ReplaceValues(replace_what=[np.nan, None], replace_with="", fields={FieldSnowplow.PAGE_REFERRER})(df_19_agg)

In [ ]:
df_afla_agg = ReclassifyInternalReferrals(site_domains=AFRO_LA.domains)(df_afla_agg)
df_dfp_agg = ReclassifyInternalReferrals(site_domains=DALLAS_FREE_PRESS.domains)(df_dfp_agg)
df_ov_agg = ReclassifyInternalReferrals(site_domains=OPEN_VALLEJO.domains)(df_ov_agg)
df_19_agg = ReclassifyInternalReferrals(site_domains=THE_19TH.domains)(df_19_agg)

## Parse internal referrer path from referrer URL

In [ ]:
df_afla_agg = ParseInternalReferrerUrls()(df_afla_agg)
df_dfp_agg = ParseInternalReferrerUrls()(df_dfp_agg)
df_ov_agg = ParseInternalReferrerUrls()(df_ov_agg)
df_19_agg = ParseInternalReferrerUrls()(df_19_agg)

In [ ]:
# CHECKPOINT 5

df_afla_agg.to_pickle(get_path_site_checkpoint(CSV_PREFIX, 5, AFRO_LA.name))
df_dfp_agg.to_pickle(get_path_site_checkpoint(CSV_PREFIX, 5, DALLAS_FREE_PRESS.name))
df_ov_agg.to_pickle(get_path_site_checkpoint(CSV_PREFIX, 5, OPEN_VALLEJO.name))
df_19_agg.to_pickle(get_path_site_checkpoint(CSV_PREFIX, 5, THE_19TH.name))

## Add session-scoped page-view indices

In [ ]:
# Read data from CHECKPOINT 5
df_afla_agg = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 5, AFRO_LA.name))
df_dfp_agg = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 5, DALLAS_FREE_PRESS.name))
df_ov_agg = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 5, OPEN_VALLEJO.name))
df_19_agg = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 5, THE_19TH.name))

In [ ]:
df_afla_agg = AddFieldSessionEventIndex()(df_afla_agg)
df_dfp_agg = AddFieldSessionEventIndex()(df_dfp_agg)
df_ov_agg = AddFieldSessionEventIndex()(df_ov_agg)
df_19_agg = AddFieldSessionEventIndex()(df_19_agg)

In [ ]:
# Check uniqueness
def test_userid_sessionidx_eventidx_uniqueness(df):
    assert (
        ~df.duplicated([FieldSnowplow.DOMAIN_USERID, FieldSnowplow.DOMAIN_SESSIONIDX, FieldNew.DOMAIN_SESSION_EVENTIDX])
    ).sum() == df.shape[0]


test_userid_sessionidx_eventidx_uniqueness(df_afla_agg)
test_userid_sessionidx_eventidx_uniqueness(df_dfp_agg)
test_userid_sessionidx_eventidx_uniqueness(df_ov_agg)
test_userid_sessionidx_eventidx_uniqueness(df_19_agg)

In [ ]:
# Check incrementing
def test_userid_sessionidx_eventidx_incrementing(df):
    def test(df2):
        s = {*df2[FieldNew.DOMAIN_SESSION_EVENTIDX]}
        assert {*range(1, len(s) + 1)} == s

    df.groupby([FieldSnowplow.DOMAIN_USERID, FieldSnowplow.DOMAIN_SESSIONIDX]).apply(test)


test_userid_sessionidx_eventidx_incrementing(df_afla_agg)
test_userid_sessionidx_eventidx_incrementing(df_dfp_agg)
test_userid_sessionidx_eventidx_incrementing(df_ov_agg)
test_userid_sessionidx_eventidx_incrementing(df_19_agg)

In [ ]:
# CHECKPOINT 6

df_afla_agg.to_pickle(get_path_site_checkpoint(CSV_PREFIX, 6, AFRO_LA.name))
df_dfp_agg.to_pickle(get_path_site_checkpoint(CSV_PREFIX, 6, DALLAS_FREE_PRESS.name))
df_ov_agg.to_pickle(get_path_site_checkpoint(CSV_PREFIX, 6, OPEN_VALLEJO.name))
df_19_agg.to_pickle(get_path_site_checkpoint(CSV_PREFIX, 6, THE_19TH.name))

## Infer target variable

In [ ]:
# Read data from CHECKPOINT 6
df_afla_agg = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 6, AFRO_LA.name))
df_dfp_agg = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 6, DALLAS_FREE_PRESS.name))
df_ov_agg = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 6, OPEN_VALLEJO.name))
df_19_agg = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 6, THE_19TH.name))

In [ ]:
df_afla_agg = SetRowIndex(
    fields_index=[FieldSnowplow.DOMAIN_USERID, FieldSnowplow.DOMAIN_SESSIONIDX, FieldNew.DOMAIN_SESSION_EVENTIDX],
    reset_index=True,
)(df_afla_agg)
df_dfp_agg = SetRowIndex(
    fields_index=[FieldSnowplow.DOMAIN_USERID, FieldSnowplow.DOMAIN_SESSIONIDX, FieldNew.DOMAIN_SESSION_EVENTIDX],
    reset_index=True,
)(df_dfp_agg)
df_ov_agg = SetRowIndex(
    fields_index=[FieldSnowplow.DOMAIN_USERID, FieldSnowplow.DOMAIN_SESSIONIDX, FieldNew.DOMAIN_SESSION_EVENTIDX],
    reset_index=True,
)(df_ov_agg)
df_19_agg = SetRowIndex(
    fields_index=[FieldSnowplow.DOMAIN_USERID, FieldSnowplow.DOMAIN_SESSIONIDX, FieldNew.DOMAIN_SESSION_EVENTIDX],
    reset_index=True,
)(df_19_agg)

In [ ]:
assert df_afla_agg.index.is_monotonic_increasing
assert df_dfp_agg.index.is_monotonic_increasing
assert df_ov_agg.index.is_monotonic_increasing
assert df_19_agg.index.is_monotonic_increasing

In [ ]:
df_afla_agg = AddFieldLeadsToNewsletterConversion(site_page_type_classifier=AFRO_LA.page_type_classifier)(df_afla_agg)
df_dfp_agg = AddFieldLeadsToNewsletterConversion(site_page_type_classifier=DALLAS_FREE_PRESS.page_type_classifier)(
    df_dfp_agg
)
df_ov_agg = AddFieldLeadsToNewsletterConversion(site_page_type_classifier=OPEN_VALLEJO.page_type_classifier)(df_ov_agg)
df_19_agg = AddFieldLeadsToNewsletterConversion(site_page_type_classifier=THE_19TH.page_type_classifier)(df_19_agg)

In [ ]:
df_afla_agg = SetRowIndex(
    fields_index=[FieldSnowplow.EVENT_ID],
    reset_index=True,
)(df_afla_agg)
df_dfp_agg = SetRowIndex(
    fields_index=[FieldSnowplow.EVENT_ID],
    reset_index=True,
)(df_dfp_agg)
df_ov_agg = SetRowIndex(
    fields_index=[FieldSnowplow.EVENT_ID],
    reset_index=True,
)(df_ov_agg)
df_19_agg = SetRowIndex(
    fields_index=[FieldSnowplow.EVENT_ID],
    reset_index=True,
)(df_19_agg)

In [ ]:
# CHECKPOINT 7

df_afla_agg.to_pickle(get_path_site_checkpoint(CSV_PREFIX, 7, AFRO_LA.name))
df_dfp_agg.to_pickle(get_path_site_checkpoint(CSV_PREFIX, 7, DALLAS_FREE_PRESS.name))
df_ov_agg.to_pickle(get_path_site_checkpoint(CSV_PREFIX, 7, OPEN_VALLEJO.name))
df_19_agg.to_pickle(get_path_site_checkpoint(CSV_PREFIX, 7, THE_19TH.name))